# TM_CBTUR Web Scraping Notebook

This notebook extracts the supervised banks list from the Turkmenistan Central Bank website and saves it to an Excel file.

In [1]:
import os
import datetime
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep

regulatorName = 'TM_CBTUR'
scriptfolder = r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\TM CBTUR"
now = datetime.datetime.now()
filename = f"{regulatorName} SQL Ready {now.strftime('%Y-%m-%d %H.%M.%S')}.xlsx"

print(f'Running {regulatorName} scraper')
print(f'Script folder: {scriptfolder}')
print(f'Output file: {filename}')

os.makedirs(scriptfolder, exist_ok=True)
os.chdir(scriptfolder)

Running TM_CBTUR scraper
Script folder: C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\TM CBTUR
Output file: TM_CBTUR SQL Ready 2026-04-16 17.26.33.xlsx


In [ ]:
regdict = {
    'TM_CBTUR 1': 'https://www.cbt.tm/en/banklar.html',
}
Typology = {
    'TM_CBTUR 1': 'Supervised Banks',
}

sqldict = {
    'bvdid': [],
    'priority': [],
    'ListLabel': [],
    'Typology': [],
    'EntryType': [],
    'Name': [],
    'InternalID_1': [],
    'InternalID_1_type': [],
    'InternalID_2': [],
    'InternalID_2_type': [],
    'InternalID_3': [],
    'InternalID_3_type': [],
    'CoType': [],
    'License_Type': [],
    'Address_1': [],
    'Address_2': [],
    'City': [],
    'Zip': [],
    'Cntry': [],
    'Phone': [],
    'Fax': [],
    'Website': [],
    'Email': [],
    'RegulationType': [],
    'RegulationTypeCode': [],
    'RegulationDate': [],
    'CancellationDate': [],
    'RegCtry': [],
    'RegCode': [],
    'ListCode': [],
    'ListLanguage': [],
    'ListValidityDate': [],
    'ListName': [],
    'ListProcessDate': [],
    'LEI Code': [],
    'BIC SWIFT Code': [],
    'Name - Mother Company': [],
    'Address_1 - Mother company': [],
    'Address_2 -  Mother company': [],
    'City - Mother company': [],
    'Zip - Mother company': [],
    'Cntry - Mother company': [],
    'Phone - Mother company': [],
    'Check': [],
}

processdate = now.strftime('%Y-%m-%d')
source_url = regdict['TM_CBTUR 1']

chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--ignore-certificate-errors')
chrome_options.add_argument('--window-size=1920,1080')

with webdriver.Chrome(options=chrome_options) as driver:
    driver.get(source_url)
    WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.ID, 'content')))
    sleep(1)
    page_html = driver.page_source

soup = BeautifulSoup(page_html, 'html.parser')
content = soup.find('div', id='content')
if content is None:
    raise ValueError('Could not find the expected <div id="content"> on the page')

for idx, bank_title in enumerate(content.find_all('h4'), start=1):
    name = bank_title.get_text(strip=True)
    block_tags = []
    for sibling in bank_title.next_siblings:
        if getattr(sibling, 'name', None) == 'h4':
            break
        if getattr(sibling, 'name', None) == 'hr':
            continue
        if isinstance(sibling, str):
            continue
        block_tags.append(sibling)

    block_html = ''.join(str(tag) for tag in block_tags)
    block_soup = BeautifulSoup(block_html, 'html.parser')

    bic = ''
    bic_tag = block_soup.find('strong')
    if bic_tag is not None:
        bic = bic_tag.get_text(strip=True)

    address = ''
    addr_tag = block_soup.find('p', class_='mail')
    if addr_tag is not None:
        address = addr_tag.get_text(strip=True)

    phone = ''
    phone_tag = block_soup.find('p', class_='phone')
    if phone_tag is not None:
        phone = phone_tag.get_text(strip=True)

    fax = ''
    fax_tag = block_soup.find('p', class_='fax')
    if fax_tag is not None:
        fax = fax_tag.get_text(strip=True)

    websites = []
    for a in block_soup.select('p.net a[href]'):
        href = a['href'].strip()
        if href.startswith('http://') or href.startswith('https://'):
            websites.append(href)
    website = ' | '.join(websites) if websites else ''

    extra_texts = []
    for p in block_soup.find_all('p'):
        label = p.get('class')
        if label not in (None, [], ['mail'], ['phone'], ['fax'], ['net']):
            extra_texts.append(p.get_text(separator=' ', strip=True))
    other_info = ' | '.join(extra_texts) if extra_texts else ''

    sqldict['bvdid'].append('')
    sqldict['priority'].append('')
    sqldict['ListLabel'].append("")
    sqldict['Typology'].append(Typology['TM_CBTUR 1'])
    sqldict['EntryType'].append('')
    sqldict['Name'].append(name)
    sqldict['InternalID_1'].append(bic)
    sqldict['InternalID_1_type'].append('BIC' if bic else '')
    sqldict['InternalID_2'].append('')
    sqldict['InternalID_2_type'].append('')
    sqldict['InternalID_3'].append('')
    sqldict['InternalID_3_type'].append('')
    sqldict['CoType'].append('')
    sqldict['License_Type'].append('')
    sqldict['Address_1'].append(address)
    sqldict['Address_2'].append('')
    sqldict['City'].append('')
    sqldict['Zip'].append('')
    sqldict['Cntry'].append('TM')
    sqldict['Phone'].append(phone)
    sqldict['Fax'].append(fax)
    sqldict['Website'].append(website)
    sqldict['Email'].append('')
    sqldict['RegulationType'].append('Regulated')
    sqldict['RegulationTypeCode'].append('')
    sqldict['RegulationDate'].append('')
    sqldict['CancellationDate'].append('')
    sqldict['RegCtry'].append('TM')
    sqldict['RegCode'].append('CBTUR')
    sqldict['ListCode'].append('1')
    sqldict['ListLanguage'].append('')
    sqldict['ListValidityDate'].append('')
    sqldict['ListName'].append(Typology['TM_CBTUR 1'])
    sqldict['ListProcessDate'].append(processdate)
    sqldict['LEI Code'].append('')
    sqldict['BIC SWIFT Code'].append(bic)
    sqldict['Name - Mother Company'].append('')
    sqldict['Address_1 - Mother company'].append('')
    sqldict['Address_2 -  Mother company'].append('')
    sqldict['City - Mother company'].append('')
    sqldict['Zip - Mother company'].append('')
    sqldict['Cntry - Mother company'].append('')
    sqldict['Phone - Mother company'].append('')
    sqldict['Check'].append('')

if not sqldict['Name']:
    raise ValueError('No bank entries were found inside <div id="content">')

df = pd.DataFrame(sqldict)
print(f'Parsed {len(df)} bank records')

df.head(10)

Parsed 9 bank records


,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,Supervised Banks,,The State Bank for Foreign Economic \n ...,BIC 390101201,BIC,,,...,,BIC 390101201,,,,,,,,
1,,,,Supervised Banks,,Joint-Stock Commercial Bank of Turkmenistan «T...,BIC 390101304,BIC,,,...,,BIC 390101304,,,,,,,,
2,,,,Supervised Banks,,The State Commercial Bank of Turkmenistan «Tur...,BIC 390101506,BIC,,,...,,BIC 390101506,,,,,,,,
3,,,,Supervised Banks,,The State Commercial Bank of Turkmenistan «Day...,BIC 390101409,BIC,,,...,,BIC 390101409,,,,,,,,
4,,,,Supervised Banks,,Joint-Stock Commercial Bank of Turkmenistan «H...,BIC 390101601,BIC,,,...,,BIC 390101601,,,,,,,,
5,,,,Supervised Banks,,The State Development Bank of Turkmenistan,BIC 390101739,BIC,,,...,,BIC 390101739,,,,,,,,
6,,,,Supervised Banks,,Joint-Stock Commercial Bank of Turkmenistan «S...,BIC 390101706,BIC,,,...,,BIC 390101706,,,,,,,,
7,,,,Supervised Banks,,Turkmen-Turkish Joint-Stock Commercial Bank,BIC 390101731,BIC,,,...,,BIC 390101731,,,,,,,,
8,,,,Supervised Banks,,"Joint-Stock Commercial Bank ""Rysgal""",BIC 390101738,BIC,,,...,,BIC 390101738,,,,,,,,


In [4]:
output_sheet = 'SQL Ready'
try:
    with pd.ExcelWriter(filename, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_urls': False}}) as writer:
        df.to_excel(writer, sheet_name=output_sheet, index=False)
except ImportError:
    df.to_excel(filename, sheet_name=output_sheet, index=False)
print(f'Saved {len(df)} rows to {filename}')

Saved 9 rows to TM_CBTUR SQL Ready 2026-04-16 17.26.33.xlsx
